In [1]:
import sys
sys.path.append('/n/home06/drooryck/codeswitching-llms')
from pathlib import Path
from july_aug_exp.src.metrics import Metrics
from july_aug_exp.src.dataset_manager import DatasetManager
from july_aug_exp.src.model_config import ModelConfig, SlurmConfig
from july_aug_exp.src.experiment import Experiment
from july_aug_exp.src.plotting import BilingualPlotter

In [2]:
repo_root = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/")

output_dir = repo_root / "results" / "sep15"
output_dir.mkdir(parents=True, exist_ok=True)

logs_dir = output_dir / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)
config = ModelConfig.load(repo_root / "configs" / "default_model.json")
slurm_config = SlurmConfig.load(repo_root / "configs" / "slurm_default.json")

slurm_config.job_name = "sep15_exp"
slurm_config.output_pattern = str(logs_dir / "slurm_%A_%a.out")
slurm_config.error_pattern = str(logs_dir / "slurm_%A_%a.err")

data_manager = DatasetManager("data", config)
metrics = Metrics("data/lexicon_new.json")
experiment = Experiment(config, data_manager, metrics, output_dir)

# Save configurations for SLURM job in results folder
config.save(output_dir / "model_config.json")
slurm_config.save(output_dir / "slurm_config.json")

props = [0.0, 0.01, 0.025, 0.05, 0.075, 0.1, 0.2, 0.3, 0.4, 0.5,
         0.6, 0.7, 0.8, 0.9, 0.925, 0.95, 0.975, 0.99, 1.0]
runs = [1, 2, 3]


## run cluster job

In [3]:
# Submit jobs to SLURM
print("Submitting jobs to SLURM...")
job_ids = experiment.submit_to_slurm(
    props=props,
    runs=runs,
    slurm_config=slurm_config,
    eval_prop=0.1
)

print(f"\nJobs submitted successfully!")
print(f"Job IDs: {job_ids}")
print(f"Results will be saved in: {output_dir}")
print(f"SLURM logs will be in: {logs_dir}")

2025-09-15 02:44:28,820 |     INFO | Submitting 57 jobs to SLURM...


Submitting jobs to SLURM...


2025-09-15 02:44:29,053 |     INFO | Submitted job array 35571584



Jobs submitted successfully!
Job IDs: ['35571584']
Results will be saved in: /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15
SLURM logs will be in: /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15/logs


## run sweep local

In [ ]:
# results_dirs = experiment.run_sweep(props=props, runs=runs, eval_prop=0.05)

# results_df = experiment.collect_results(results_dirs)
# experiment.save_summary(results_df)
# experiment.create_plots(results_df)

2025-09-12 18:37:07,637 |     INFO | Starting experiment: prop=0.0, run_id=1
2025-09-12 18:37:07,638 |     INFO | Loading data...
2025-09-12 18:37:07,988 |     INFO | Train set: 10440 rows (0 FR, 10440 NL)
2025-09-12 18:37:07,989 |     INFO | Train set: 10440 samples
2025-09-12 18:37:07,991 |     INFO | Test set: 259 samples
2025-09-12 18:37:07,994 |     INFO | Creating tokenizer and datasets...
2025-09-12 18:37:08,857 |     INFO | Creating model...
2025-09-12 18:37:09,048 |     INFO | Creating trainer...
2025-09-12 18:37:09,096 |  WARNING | Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2025-09-12 18:37:09,101 |     INFO | Starting training...
/n/home06/drooryck/envs/codeswitching-py310/lib/python3.10/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pin

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## plot

In [5]:
from pathlib import Path

output_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep12.5")
run_dirs = list(output_dir.glob("p*_run*"))
results_df = experiment.collect_results(run_dirs)
experiment.save_summary(results_df)

plotter = BilingualPlotter(results_df, output_dir / "plots")
plotter.print_metrics_summary()
plotter.create_all_plots()

2025-09-15 01:40:45,644 |     INFO | Summary saved to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep12.5/summary.csv
2025-09-15 01:40:45,673 |     INFO | Creating exact match plot...
/n/home06/drooryck/codeswitching-llms/july_aug_exp/src/plotting.py:118: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar='sd'` for the same effect.

  sns.lineplot(
/n/home06/drooryck/codeswitching-llms/july_aug_exp/src/plotting.py:118: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar='sd'` for the same effect.

  sns.lineplot(
/n/home06/drooryck/codeswitching-llms/july_aug_exp/src/plotting.py:118: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar='sd'` for the same effect.

  sns.lineplot(



Available metrics:
['prop', 'run_id', 'run_dir', 'fr_exact', 'fr_avg_fr', 'fr_avg_nl', 'fr_part_final', 'nl_exact', 'nl_avg_fr', 'nl_avg_nl', 'nl_part_final', 'overall_exact']

Runs per proportion:
prop
0.00    3
0.01    3
0.02    3
0.05    3
0.07    3
0.10    3
0.20    3
0.30    3
0.40    3
0.50    3
0.60    3
0.70    3
0.80    3
0.90    3
0.92    3
0.95    3
0.97    3
0.99    3
1.00    3
dtype: int64

Summary statistics:
            prop     run_id   fr_exact  fr_avg_fr  fr_avg_nl  fr_part_final  \
count  57.000000  57.000000  57.000000  57.000000  57.000000           57.0   
mean    0.498947   2.000000   0.856961   0.942916   0.057084            0.0   
std     0.387145   0.823754   0.296105   0.226115   0.226115            0.0   
min     0.000000   1.000000   0.000000   0.000000   0.000000            0.0   
25%     0.070000   1.000000   0.932836   1.000000   0.000000            0.0   
50%     0.500000   2.000000   0.992537   1.000000   0.000000            0.0   
75%     0.920000   

2025-09-15 01:40:46,124 |     INFO | Saved plot to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep12.5/plots/exact_match.png
2025-09-15 01:40:46,125 |     INFO | Creating word order plot...
/n/home06/drooryck/codeswitching-llms/july_aug_exp/src/plotting.py:118: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar='sd'` for the same effect.

  sns.lineplot(
/n/home06/drooryck/codeswitching-llms/july_aug_exp/src/plotting.py:118: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar='sd'` for the same effect.

  sns.lineplot(
2025-09-15 01:40:46,449 |     INFO | Saved plot to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep12.5/plots/word_order.png
2025-09-15 01:40:46,451 |     INFO | Creating French token distribution plot...
/n/home06/drooryck/codeswitching-llms/july_aug_exp/src/plotting.py:118: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar='sd'` for the same effect.

  sns.lineplot(
/n/home06/drooryck/codeswitching-